## Instalações

In [ ]:
%pip install -U langchain-openai
%pip install -U langchain-community
%pip install -U langgraph

## Importações

In [1]:
import os
from dotenv import load_dotenv
import operator

In [2]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List

# langchain
from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage, BaseMessage
from langchain_community.tools.tavily_search import TavilySearchResults

/tmp/ipykernel_74360/3350279336.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


## Configuração de Variáveis de Ambiente

In [3]:
load_dotenv()

api_key = os.getenv('OPENAI_API_KEY')
gpt_llm_model = os.getenv('GPT_LLM_MODEL')
tavily_api_key = os.getenv('TAVILY_API_KEY')

### Teste de Ferramentas (Tavily)

In [4]:
tool = TavilySearchResults(max_results=4)
print(type(tool))
print(tool.name)

<class 'langchain_community.tools.tavily_search.tool.TavilySearchResults'>
tavily_search_results_json


/tmp/ipykernel_74360/1287549681.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool = TavilySearchResults(max_results=4)


## Definição do Estado do Agente (AgentState)

In [ ]:
class AgentState(TypedDict):
  messages: Annotated[list[AnyMessage], operator.add]

## Implementação da Classe Agent

In [ ]:
class Agent:
    def __init__(self, model, tools, system=""):

        # Guardando o prompt de sistema
        self.system = system

        # Criando o grafo
        graph = StateGraph(AgentState)

        # Adicionando os nós
        graph.add_node("llm", self.call_gemini)
        graph.add_node("action", self.take_action)

        # Criando a decisão condicional
        # Essa parte determina o que acontece depois que o GPT responde.
        # O LangGraph executa exists_action() e observa seu retorno
        # Isso permite que o modelo responda diretamente ou solicite uma ferramenta.
        graph.add_conditional_edges(
            "llm", self.exists_action, {True: "action", False: END}
        )

        # Criando o retorno ao modelo
        # Sempre que o nó action terminar, execute novamente o nó llm.
        graph.add_edge("action", "llm")

        # Definindo o ponto de entrada
        graph.set_entry_point("llm")

        # Compilando o grafo
        self.graph = graph.compile()

        # Registrando as ferramentas
        self.tools = {t.name: t for t in tools}

        # Associando ferramentas ao modelo
        # Ela informa ao GPT quais ferramentas estão disponíveis e como solicitá-las.
        # O modelo passa a conhecer as descrições e os parâmetros das ferramentas.
        # Ele configura o modelo para produzir chamadas de ferramentas quando necessário.
        self.model = model.bind_tools(tools)

    def exists_action(self, state: AgentState):

        # Pega a última mensagem do histórico.
        result = state["messages"][-1]

        # Verifica se o modelo solicitou alguma ferramenta.
        # Se o GPT simplesmente responder "Olá!", a lista tool_calls estará vazia e a função retornará False.
        return len(result.tool_calls) > 0

    def call_gemini(self, state: AgentState):

        # Recupera o histórico:
        messages = state["messages"]

        # Adiciona a instrução de sistema, se existir
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages

        # Envia as mensagens ao GPT
        # O modelo pode retornar uma resposta normal ou uma solicitação de ferramenta.
        message = self.model.invoke(messages)

        # Retorna a mensagem para atualizar o estado
        return {"messages": [message]}

    def take_action(self, state: AgentState):

        # Recuperando as chamadas
        # Recupera todas as ferramentas que o GPT solicitou na última resposta.
        tool_calls = state["messages"][-1].tool_calls

        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            if not t["name"] in self.tools:
                print(f"\n bad tool name....")
                result = "bad tool name, retry"
            else:
                result = self.tools[t["name"]].invoke(t["args"])
                
            # Devolvendo o resultado ao GPT
            results.append(
                ToolMessage(tool_call_id=t["id"], name=t["name"], content=str(result))
            )
        print("Back to the model!")
        
        # Os resultados são adicionados ao estado e o grafo retorna ao nó llm.
        return {"messages": results}